# Deploy Sarvam **Saaras v3.1** Speech-to-Text Model Package from AWS Marketplace

This sample notebook shows you how to deploy [Saaras v3.1](https://aws.amazon.com/marketplace/) <!-- TODO: replace with the marketplace listing URL --> — Sarvam AI's speech-to-text model for Indian languages and English — using Amazon SageMaker, and how to run all three supported inference modes:

1. **Real-time inference** — POST a `multipart/form-data` request with an audio file to a SageMaker endpoint and get the full transcript back as JSON (optionally with timestamps).
2. **Real-time streaming inference** — stream microphone-style audio chunks over SageMaker's bidirectional HTTP/2 streaming API and receive voice-activity events and transcripts as they are produced.
3. **Batch transform** — transcribe a folder of pre-serialized payloads from S3 in one offline job.

> **Note**: This is a reference notebook and it cannot run unless you make the changes suggested in the notebook (at minimum, set `model_package_arn`).

## Pre-requisites

1. This notebook contains elements which render correctly in the Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that the IAM role used has **AmazonSageMakerFullAccess**.
1. To deploy this ML model successfully, ensure that one of the following applies:
    1. Your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used:
        1. **aws-marketplace:ViewSubscriptions**
        1. **aws-marketplace:Unsubscribe**
        1. **aws-marketplace:Subscribe**
    1. or your AWS account has a subscription to the listing. If so, skip step: [Subscribe to the model package](#1.-Subscribe-to-the-model-package).
1. Real-time inference uses one `ml.g6e.xlarge` instance (1× NVIDIA L40S GPU) — make sure your account has quota for `ml.g6e.xlarge for endpoint usage`. Batch transform additionally needs `ml.g6.xlarge for transform job usage` quota (transform quotas default to 0 in new accounts; request an increase via Service Quotas / AWS Support).
1. The notebook works with both major versions of the SageMaker Python SDK — v3 (preinstalled on current SageMaker Studio images) and v2. All SageMaker API calls are made through `boto3`.

## Contents

1. [Subscribe to the model package](#1.-Subscribe-to-the-model-package)
2. [Create an endpoint and perform real-time inference](#2.-Create-an-endpoint-and-perform-real-time-inference)
3. [Real-time streaming inference](#3.-Real-time-streaming-inference-(bidirectional))
4. [Perform batch inference](#4.-Perform-batch-inference)
5. [Clean-up](#5.-Clean-up)

## Usage instructions

You can run this notebook one cell at a time (by using Shift+Enter for running a cell).

## 1. Subscribe to the model package

To subscribe to the model package:
1. Open the model package listing page [Saaras v3.1](https://aws.amazon.com/marketplace/). <!-- TODO: replace with the marketplace listing URL -->
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agree with EULA, pricing, and support terms.
1. Once you click on the **Continue to configuration** button and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model using boto3. Copy the ARN corresponding to your region and specify it in the following cell.

In [ ]:
model_package_arn = "<Customer to specify Model package ARN corresponding to their AWS region>"

### Set up the SageMaker session and execution role

The next cell prepares everything the notebook needs to talk to SageMaker:

- **SageMaker execution role** — the IAM role that SageMaker assumes when acting on your behalf: pulling the model container image, downloading the packaged model weights, writing CloudWatch logs, and running the endpoints and transform jobs created below (it is passed as `ExecutionRoleArn` to `create_model`).
  - Running **inside SageMaker Studio or a SageMaker Notebook Instance** (recommended): leave `sagemaker_execution_role_arn = None` — `get_execution_role()` automatically resolves the execution role attached to this notebook environment.
  - Running **anywhere else** (laptop, EC2, CI): `get_execution_role()` cannot resolve a role there, so set `sagemaker_execution_role_arn` to the ARN of a SageMaker execution role in your account, e.g. `arn:aws:iam::<account-id>:role/service-role/AmazonSageMaker-ExecutionRole-...` (IAM console → Roles). The role must be assumable by `sagemaker.amazonaws.com` and have `AmazonSageMakerFullAccess` (or equivalent), and the credentials running this notebook need `iam:PassRole` on that role.
- **Session and clients** — the `Session` provides the AWS region and a default S3 bucket (`sagemaker-<region>-<account-id>`, created on first use) that the batch-transform section uses for input/output data; the `boto3` clients are used for all SageMaker API calls.

In [ ]:
import base64
import json
import mimetypes
import os
import uuid
import wave

import boto3

try:  # SageMaker Python SDK v3
    from sagemaker.core.helper.session_helper import Session, get_execution_role
except ImportError:  # SageMaker Python SDK v2
    from sagemaker import Session, get_execution_role

# Leave as None inside SageMaker Studio / Notebook Instances (auto-detected).
# Set to your SageMaker execution role ARN when running anywhere else, e.g.
# "arn:aws:iam::<account-id>:role/service-role/AmazonSageMaker-ExecutionRole-XXXXXXXXXXXXXXX"
sagemaker_execution_role_arn = None

role = sagemaker_execution_role_arn or get_execution_role()
sagemaker_session = Session()
bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name
runtime = boto3.client("sagemaker-runtime", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print("Execution role:", role)
print("Region        :", region)
print("Default bucket:", bucket)

The next cell defines the names and instance types used throughout the notebook.

> **Important — model identifier**: in API payloads the model is addressed by its family identifier **`saaras:v3`** (the `model` form field / query parameter). The container rejects other values, including `saaras:v3.1` — the v3.1 revision is what the model package ships, not a separate API identifier.

In [ ]:
model_name = "saaras-v3-1-speech-to-text"

# Model identifier expected by the container in API payloads (NOT "saaras:v3.1").
SAARAS_MODEL_ID = "saaras:v3"

real_time_inference_instance_type = "ml.g6e.xlarge"  # 1x NVIDIA L40S GPU
batch_transform_inference_instance_type = "ml.g6.xlarge"

## 2. Create an endpoint and perform real-time inference

If you want to understand how real-time inference with Amazon SageMaker works, see the [documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).

### A. Create an endpoint

In [ ]:
# Create a deployable model from the model package.
sm_client.create_model(
    ModelName=model_name,
    PrimaryContainer={"ModelPackageName": model_package_arn},
    ExecutionRoleArn=role,
    EnableNetworkIsolation=True,  # marketplace model packages run network-isolated
)

# Deploy the model. Endpoint creation takes roughly 10-15 minutes
# (GPU instance provisioning + container image + model weights download).
sm_client.create_endpoint_config(
    EndpointConfigName=model_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InstanceType": real_time_inference_instance_type,
            "InitialInstanceCount": 1,
        }
    ],
)
sm_client.create_endpoint(EndpointName=model_name, EndpointConfigName=model_name)
print(f"Creating endpoint {model_name} (takes roughly 10-15 minutes)...")
sm_client.get_waiter("endpoint_in_service").wait(EndpointName=model_name)
print("Endpoint is InService")

Once the endpoint has been created, you can perform real-time inference.

### B. Create the input payload

The endpoint accepts a **`multipart/form-data`** request body — the same shape as a `curl --form` upload — with the following parts:

| Form field | Required | Description |
|---|---|---|
| `file` | yes | The audio file to transcribe (e.g. a WAV file). |
| `model` | yes | Model identifier: `saaras:v3`. |
| `with_timestamps` | no | `"true"` to include segment timestamps in the response. |

Because `boto3`'s `invoke_endpoint` takes raw bytes, we serialize the multipart body ourselves and pass the matching `Content-Type` header (which carries the multipart boundary). The sample audio (`data/input/real-time/sample_audio.wav`) is an 18.7-second English clip.

In [ ]:
def build_multipart_body(fields: dict, file_path: str, boundary: str) -> bytes:
    """Serialize simple form fields plus one audio file into a multipart/form-data body."""
    crlf = b"\r\n"
    parts = []

    for name, value in fields.items():
        parts.append(f"--{boundary}".encode())
        parts.append(f'Content-Disposition: form-data; name="{name}"'.encode())
        parts.append(b"")
        parts.append(str(value).encode())

    filename = os.path.basename(file_path)
    file_content_type = mimetypes.guess_type(filename)[0] or "application/octet-stream"
    with open(file_path, "rb") as fh:
        file_bytes = fh.read()

    parts.append(f"--{boundary}".encode())
    parts.append(
        f'Content-Disposition: form-data; name="file"; filename="{filename}"'.encode()
    )
    parts.append(f"Content-Type: {file_content_type}".encode())
    parts.append(b"")

    body = crlf.join(parts) + crlf + file_bytes + crlf
    body += f"--{boundary}--".encode() + crlf
    return body


audio_path = "data/input/real-time/sample_audio.wav"
boundary = f"----sagemakerformboundary{uuid.uuid4().hex}"

payload = build_multipart_body(
    {"model": SAARAS_MODEL_ID, "with_timestamps": "true"}, audio_path, boundary
)
content_type = f"multipart/form-data; boundary={boundary}"
print(f"Payload: {len(payload)} bytes\nContent-Type: {content_type}")

### C. Perform real-time inference

In [ ]:
response = runtime.invoke_endpoint(
    EndpointName=model_name,
    ContentType=content_type,
    Accept="application/json",
    Body=payload,
)
result = json.loads(response["Body"].read())
result

### D. Visualize output

The response is a JSON document (see `data/output/real-time/sample_response.json` for the expected output of the sample clip):

| Key | Description |
|---|---|
| `request_id` | Unique identifier of the request. |
| `transcript` | The full transcript text. |
| `language_code` | Detected language (BCP-47, e.g. `en-IN`). |
| `language_probability` | Confidence of the language detection. |
| `timestamps` | Present when `with_timestamps=true`: parallel lists `words`, `start_time_seconds`, `end_time_seconds`. |

In [ ]:
print("Transcript:\n" + result["transcript"])
print("\nDetected language:", result["language_code"],
      f"(p={result['language_probability']})")

if "timestamps" in result:
    ts = result["timestamps"]
    print("\nSegments:")
    for text, t0, t1 in zip(ts["words"], ts["start_time_seconds"], ts["end_time_seconds"]):
        print(f"  [{t0:7.2f}s - {t1:7.2f}s] {text}")

## 3. Real-time streaming inference (bidirectional)

The same endpoint also supports **bidirectional streaming**: you send small audio chunks as you capture them and receive voice-activity events and transcripts back on the same connection — ideal for live microphone or telephony audio.

SageMaker does not expose a raw WebSocket URL. Bidirectional streaming goes over a SigV4-signed **HTTP/2** connection using the [`aws-sdk-sagemaker-runtime-http2`](https://pypi.org/project/aws-sdk-sagemaker-runtime-http2/) client and the `InvokeEndpointWithBidirectionalStream` API. Over that transport, the model speaks a simple JSON message protocol:

**Client → server** (each message is one JSON text frame):

```
{"type": "config"}                                                       # 1. open the session
{"audio": {"data": "<base64 PCM>", "sample_rate": 16000,
           "encoding": "audio/wav"}}                                     # 2. one message per ~100 ms chunk
{"type": "flush"}                                                        # 3. finalize the transcript
```

**Server → client**:

```
{"type": "events", "data": {"signal_type": "START_SPEECH" | "END_SPEECH", ...}}   # VAD events
{"type": "data",   "data": {"transcript": "...", "language_code": "...", ...}}    # transcripts
```

Session parameters are passed in the **query string** of the streaming request (`ModelQueryString`), not in the `config` message:

| Query parameter | Description |
|---|---|
| `model` | `saaras:v3` — **must be URL-encoded** (SageMaker rejects a literal `:`). |
| `language-code` | Language of the audio, e.g. `en-IN`. |
| `sample_rate` | PCM sample rate; the streaming endpoint accepts **8000 or 16000** only. |
| `vad_signals` | `true` to receive `START_SPEECH` / `END_SPEECH` events. |

Three protocol details matter (handled by the code below):

1. Every payload part must be sent with **`data_type="UTF8"`** so the model receives *text* frames — binary frames are rejected.
2. The query string values must be **URL-encoded**.
3. After sending `flush`, **keep the input stream open** until the final transcript arrives — closing it immediately tears down the whole bidirectional stream.

Expected messages for the sample clip are in `data/output/streaming/sample_messages.jsonl`.

In [ ]:
%pip install --quiet aws-sdk-sagemaker-runtime-http2

In [ ]:
import asyncio
import time
from urllib.parse import urlencode

from aws_sdk_sagemaker_runtime_http2.client import SageMakerRuntimeHTTP2Client
from aws_sdk_sagemaker_runtime_http2.config import Config, HTTPAuthSchemeResolver
from aws_sdk_sagemaker_runtime_http2.models import (
    InvokeEndpointWithBidirectionalStreamInput,
    RequestPayloadPart,
    RequestStreamEventPayloadPart,
)
from smithy_aws_core.auth.sigv4 import SigV4AuthScheme
from smithy_aws_core.identity import EnvironmentCredentialsResolver

# The HTTP/2 SDK reads credentials from environment variables, so export the
# credentials boto3 resolved (works with execution roles, profiles, and SSO).
creds = boto3.Session().get_credentials().get_frozen_credentials()
os.environ["AWS_ACCESS_KEY_ID"] = creds.access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = creds.secret_key
if creds.token:
    os.environ["AWS_SESSION_TOKEN"] = creds.token
os.environ.setdefault("AWS_REGION", region)

Load the streaming sample audio. It is the same clip already resampled to **16 kHz mono 16-bit PCM** (`data/input/streaming/sample_audio_16k.wav`) — when streaming your own audio, resample it to 8 kHz or 16 kHz first.

In [ ]:
STREAM_SAMPLE_RATE = 16000
CHUNK_BYTES = STREAM_SAMPLE_RATE * 2 // 10  # 100 ms of 16-bit mono PCM = 3200 bytes
CHUNK_INTERVAL = 0.1  # pace the sends to simulate a live audio source

with wave.open("data/input/streaming/sample_audio_16k.wav", "rb") as wav:
    assert wav.getframerate() == STREAM_SAMPLE_RATE and wav.getnchannels() == 1
    pcm = wav.readframes(wav.getnframes())

print(f"{len(pcm)} bytes PCM (~{len(pcm) / (STREAM_SAMPLE_RATE * 2):.2f}s @16kHz mono)")

In [ ]:
async def stream_transcribe(
    endpoint_name: str,
    pcm: bytes,
    language_code: str = "en-IN",
    idle_timeout: float = 10.0,
    hard_timeout: float = 60.0,
) -> list:
    """Stream PCM audio to the endpoint and return the received JSON messages."""
    config = Config(
        endpoint_uri=f"https://runtime.sagemaker.{region}.amazonaws.com:8443",
        region=region,
        aws_credentials_identity_resolver=EnvironmentCredentialsResolver(),
        auth_scheme_resolver=HTTPAuthSchemeResolver(),
        auth_schemes={"aws.auth#sigv4": SigV4AuthScheme(service="sagemaker")},
    )
    client = SageMakerRuntimeHTTP2Client(config=config)

    # Session parameters travel in the query string and MUST be URL-encoded.
    query = urlencode(
        {
            "model": SAARAS_MODEL_ID,
            "language-code": language_code,
            "sample_rate": STREAM_SAMPLE_RATE,
            "vad_signals": "true",
        }
    )

    stream = await client.invoke_endpoint_with_bidirectional_stream(
        InvokeEndpointWithBidirectionalStreamInput(
            endpoint_name=endpoint_name, model_query_string=query
        )
    )
    _, output_stream = await stream.await_output()

    messages = []
    state = {"last_msg": time.monotonic()}

    async def send_json(obj: dict) -> None:
        # data_type="UTF8" is required: the model only accepts text frames.
        part = RequestPayloadPart(bytes_=json.dumps(obj).encode(), data_type="UTF8")
        await stream.input_stream.send(RequestStreamEventPayloadPart(value=part))

    async def receiver() -> None:
        while True:
            result = await output_stream.receive()
            if result is None:  # server closed the output stream
                break
            value = getattr(result, "value", None)
            raw = getattr(value, "bytes_", None) if value else None
            if not raw:
                continue
            state["last_msg"] = time.monotonic()
            msg = json.loads(raw)
            messages.append(msg)
            print("<<", json.dumps(msg, ensure_ascii=False)[:160])
            if msg.get("type") == "data":  # final transcript received
                break

    recv_task = asyncio.create_task(receiver())

    await send_json({"type": "config"})
    for i in range(0, len(pcm), CHUNK_BYTES):
        chunk = pcm[i : i + CHUNK_BYTES]
        await send_json(
            {
                "audio": {
                    "data": base64.b64encode(chunk).decode(),
                    "sample_rate": STREAM_SAMPLE_RATE,
                    "encoding": "audio/wav",
                }
            }
        )
        await asyncio.sleep(CHUNK_INTERVAL)
    await send_json({"type": "flush"})

    # Keep the input stream OPEN after flush; wait for the final transcript.
    flush_at = time.monotonic()
    state["last_msg"] = flush_at  # measure idle time from the flush onwards
    while not recv_task.done():
        await asyncio.sleep(0.25)
        now = time.monotonic()
        if now - flush_at > hard_timeout or now - state["last_msg"] > idle_timeout:
            recv_task.cancel()
            break
    try:
        await recv_task
    except asyncio.CancelledError:
        pass
    await stream.input_stream.close()
    return messages


# Jupyter runs an asyncio event loop, so the coroutine can be awaited directly.
messages = await stream_transcribe(model_name, pcm)

In [ ]:
transcripts = [m["data"]["transcript"] for m in messages if m.get("type") == "data"]
print("Streaming transcript:\n" + "\n".join(transcripts))

### Delete the endpoint

Now that you have successfully performed both real-time inference modes, you do not need the endpoint any more. You can terminate the endpoint to avoid being charged.

In [ ]:
sm_client.delete_endpoint(EndpointName=model_name)
sm_client.delete_endpoint_config(EndpointConfigName=model_name)

## 4. Perform batch inference

In this section, you will perform batch inference using multiple input payloads together. If you are not familiar with batch transform, and want to learn more, see these links:
1. [How it works](https://docs.aws.amazon.com/sagemaker/latest/dg/ex1-batch-transform.html)
2. [How to run a batch transform job](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-batch.html)

Batch transform sends each S3 object **as-is** as one request body. The model expects `multipart/form-data`, so each input file must be a fully serialized multipart body, and — because the `Content-Type` of the job must match the bodies — all files must share one **fixed boundary**. The cell below builds such a payload from the sample WAV (a ready-made copy ships in `data/input/batch/payload.multipart`).

> **Quota note**: transform-job quotas default to 0 in new accounts. Request `ml.g6.xlarge for transform job usage` ≥ 1 before running this section. Also keep each payload under the job's `MaxPayloadInMB`.

In [ ]:
BATCH_BOUNDARY = "sarvamsttsampleboundary"
batch_content_type = f"multipart/form-data; boundary={BATCH_BOUNDARY}"

os.makedirs("data/input/batch", exist_ok=True)
batch_payload = build_multipart_body(
    {"model": SAARAS_MODEL_ID, "with_timestamps": "true"},
    "data/input/real-time/sample_audio.wav",
    BATCH_BOUNDARY,
)
with open("data/input/batch/payload.multipart", "wb") as f:
    f.write(batch_payload)

# Upload the batch-transform job input files to S3
transform_input = sagemaker_session.upload_data("data/input/batch", key_prefix=model_name)
print("Transform input uploaded to " + transform_input)

In [ ]:
# Run the batch-transform job
transform_job_name = f"{model_name}-{uuid.uuid4().hex[:8]}"
transform_output = f"s3://{bucket}/{model_name}/batch-output"

sm_client.create_transform_job(
    TransformJobName=transform_job_name,
    ModelName=model_name,
    MaxPayloadInMB=6,
    TransformInput={
        "DataSource": {
            "S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": transform_input}
        },
        "ContentType": batch_content_type,
    },
    TransformOutput={"S3OutputPath": transform_output, "Accept": "application/json"},
    TransformResources={
        "InstanceType": batch_transform_inference_instance_type,
        "InstanceCount": 1,
    },
)
print(f"Waiting for transform job {transform_job_name}...")
sm_client.get_waiter("transform_job_completed_or_stopped").wait(
    TransformJobName=transform_job_name
)
print("Status:", sm_client.describe_transform_job(
    TransformJobName=transform_job_name)["TransformJobStatus"])

# Output is available on the following path
transform_output

Each input object produces one `<object name>.out` JSON file under the output path — the same response schema as real-time inference (expected output: `data/output/batch/payload.multipart.out`).

In [ ]:
out_bucket, _, out_prefix = transform_output[len("s3://") :].partition("/")
obj = boto3.client("s3").get_object(
    Bucket=out_bucket, Key=f"{out_prefix}/payload.multipart.out"
)
print(json.dumps(json.loads(obj["Body"].read()), indent=2, ensure_ascii=False))

## 5. Clean-up

### A. Delete the model

In [ ]:
sm_client.delete_model(ModelName=model_name)

### B. Unsubscribe to the listing (optional)

If you would like to unsubscribe to the model package, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. Note — you can find this information by looking at the container name associated with the model.

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to the __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml).
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__ to cancel the subscription.